# Bayesian Regression with Linear Basis Function Models

## Linear basis function models

Linear regression models are linear in their parameters, but not necessarily in their input variables. This lets them model any non-linearities from input variables to targets (eg. polynomial regression, shown later). 

A linear regression model y(x,w) can be defined as:

$y(x, w) = w_0 + \sum^{M-1}_{j=1} w_j{\phi}_j(x) = \sum^{M-1}_{j=0} w_j{\phi}_j(x) = w^T{\phi}(x)$

where ${\phi}_j$ are basis functions and $M$ is the number of parameters $w_j$ including the bias term $w_0$. 

Here we use the convention ${\phi}_0(x) = 1$.

In the simplest case of linear regression models, they also use linear functions from their input variables: the set of basis functions is the identity ${\phi}(x) = x$

The target variable $t$ of an observation $x$ is given by a deterministic function $y(x,w)$ plus adding random noise $\epsilon$:

$t = y(x,w) + \epsilon$

- We assume that the noise is normally distributed: has zero mean and precision (= inverse variance) $\beta$. The probabilistic model (conditional distribution of $t$ given $x$) is therefore:

$p(t | x,w,\beta) = N(t | y(x,w),{\beta}^{-1}) = \sqrt{{\beta}/2{\pi}} exp(-({\beta}/2)(t - y(x,w))^2)$

The mean of this distribution is the regression function $y(x,w)$

### Basically,

What that means is **the data is a curve plus some random wobble**.

- y(x,w) is the "true" smooth curve: the weighted sum of basis functions. It's deterministic: any x,w gives the same y.

- t is what you actually measure at that x: the dot in the scatter plot.

- ${\epsilon}$ is why the dot doesn't sit exactly on the curve -- measurements are messy.

So dot = curve + wobble.

"Noise is normally distributed with zero mean" means the wobble follows a bell curve centered on 0. Dots are equally likely to be above/below the curve. No bias. Normally distributed means small wobbles are common and big ones are rare.

Precision $\beta$ is how tight the wobble is:
- variance ${\sigma}^2$ = how spread out the noise is
- precision ${\beta} = 1/{\sigma}^2$ = how concentrated the noise is

Big $\beta$ means the dots hug the curve; small means they scatter widely. Precision is used because it makes the algebra tighter; $\beta^{-1}$ = variance.

**The "conditional distribution of t given x"**, $p(t | x,w,{\beta})$: *if I already know the input x, weights w, and noise level ${\beta}$, how likely is each posible measured value t?*

- Since t = curve + bell-shaped noise, the answer is a bell curved centered on y(x,w) with variance $1/\beta$.
- The big formula is the standard bell-curve (Gaussian) formula with those values plugged in. It says "values of t near y(x,w) are likely, and values far are unlikely.

If you measured the same x many times, your dots would average out to y(x,w). The curve is your best guess, and the bell describes the scatter around it.

This setup matters for the next parts, because it lets you score a candidate w: "given the curve had these weights, how likely are the dots I saw?" = the **likelihood**. And then comes the Bayesian step: combine it with a **prior** (belief about w before seeing data) to get a **posterior** (beleif after seeing).

## Likelihood function

**Remember the big question** we're trying to answer:

we have some noisy measurements, and we want to know which curve produced them, plus how sure we can be about that.

Say we're tracking how an apartment's temperature (t) changes over time of day (x) (hours). We take 10 readings $x_1,...,x_{10}$ : $t_1,...,t_{10}$.
- $X$ is all our inputs stacked together: the 10 hours
- $t$ is all our measurements: the 10 temperatures.

We believe there's a smooth daily temperature curve, so each reading is that curve plus a bit of thermometer wobble. The curve is set by the weights $w$. **we never get to see $w$; we only see the dots**.

So in this section, we want to work backwards from the dots to get $w$.

- The **likelihood** answers one question: **"If the weights were w, how plausible would our 10 readings be?"**
    - Try a w. Let's say that w's curve passes close to all 10 dots. Each would need a small, typical wobble (to touch the curve), so the readings are plausible and we conclude likelihood is high.
    - Try a w. Let's say its curve sits 5 degrees above everything. Every dot would need a big, rare wobble to touch the curve. So likelihood is low.

Likelihood therefore works like a **scoring function for candidate curves**: how well does this curve explain what we saw?

-----

For model fitting and model parameter inference, we use a training set of N independent, identically distributed (i.i.d) observations: $x_1,...,x_N$ and corresponding targets: $t_1,...,t_N$.

We combine the column vectors $x_i$ into matrix X, where $X_{i,:} = x^T_i$, and scalar targets $t_i$ into column vector $t$.

The joint conditional probability of targets $t$ given $X$ can be formulated as:

$p(t|X,w,\beta) = \prod_{i=1}^N \mathcal{N}(t_i | w^T \phi(x_i),{\beta}^{-1})$

For better readability, this function will be written as $p(t | w,\beta)$ instead of $p(t|X,w,\beta)$ from now on. The log likelihood of this function:

$\log p(t|w,\beta) = N/2 \log \beta - N/2 \log 2\pi - \beta E_D(w)$

- Where E_D(w) is the sum-of-squares error function coming from the exponent of the likelihood function:

$E_D(w) = 1/2 \sum^N_{i=1} (t_i - w^T \phi (x_i))^2 = 1/2 ||t-\boldsymbol\Phi w||^2$

- Matrix $\boldsymbol\Phi$ is the *"design matrix"* (NxM basis function matrix), defined as

$$
\boldsymbol\Phi = 
\begin{pmatrix}
\phi_0(\mathbf{x}_1) &  \phi_1(\mathbf{x}_1) & \cdots & \phi_{M-1}(\mathbf{x}_1) \\ 
\phi_0(\mathbf{x}_2) &  \phi_1(\mathbf{x}_2) & \cdots & \phi_{M-1}(\mathbf{x}_2) \\
\vdots & \vdots & \ddots & \vdots \\
\phi_0(\mathbf{x}_N) &  \phi_1(\mathbf{x}_N) & \cdots & \phi_{M-1}(\mathbf{x}_N)
\end{pmatrix} \tag{7}
$$


Simpler:

1. For each reading i, compute the bell-curve score (last section): how close is $t_i$ to the curve's height at $x_i$?
2. Multiply the N scores together. The noise is assumed independent across readings, so they just multiply; a product is high only if *every* dot fits reasonably well.
3. Taking the log turns the **product** into a **sum**, and the only part that depends on w is $-\beta E_D(w)$. So the w with the highest likelihood is the one with smallest sum of squared errors. Maximum likelihood here is least squares.

After this section... 
- **Non-bayesian route**: we'll be able to pick the single w with the highest score (maximum likelihood; here ordinary least squares). We'll get on best curve.
- **Bayesian route (this notebook)**: don't pick just one, but combine the score with our **prior** to get the **posterior**, a probability for every possible w. We'll end up with a best-guess curve plus an uncertainty band. With the thermometer example, this would be confident around the hours we measured and say "not sure" at a time where have no readings. 

That uncertainty band is the whole payoff we're building towards.

Pipeline of what we do:

prior over w → (× likelihood of the readings) → posterior over w → (average over curves) → distribution of temperature at any hour

==

combine the likelihood for all candidate ws with our prior over w to get a posterior over w. Then use that to get the distribution of temperatures at any given hour.

## Maximum likelihood 

Maximizing log likelihood (= minimizing the sum-of-squares error function) wrt **w** gives the maximum likelihood estimate of parameters **w**. 

Maximum likelihood estimation can lead to severe over-fitting if complex models are fit to datasets of limited size. A common approach to prevent this is adding a regularization term (seen in makemore section) over the error function. We will see shortly that **this regularization term arises naturally** when following a Bayesian approach (when defining a prior distribution over parameters w).

## Bayesian approach

Now let's do the bayesian approach.

### Prior and posterior distribution

For a Bayesian treatment of linear regression, we need a prior probability distribution over model parameters $w$. For simplicity, we will use an isotropic Gaussian distribution (def below) over parameters $w$ with zero mean. Prior:

$$p(\mathbf{w} \lvert \alpha) = \mathcal{N}(\mathbf{w} \lvert \mathbf{0}, \alpha^{-1}\mathbf{I}) \tag{8}$$

An isotropic Gaussian distribution has a diagonal covariance matrix where all diagonal elements have the same variance $\alpha^{-1}$ ($\alpha$ is the precision of the prior). 

This formula says "before seeing any data, I think each weight is probably near 0, and I'm equally unsure about all of them in the same way."

A **covariance matrix** is a bell curve in many dimensions. With one number we only need a variance (how spread out our guess is). But with several ws ($w_0,...$) we need to describe two things:
1. How unsure are we about each weight on its own? These are the variances on the diagonal of the matrix.
2. Do the weights move together? For example, "if $w_1$ is big, $w_2$ is probably big". These are the covariances, off the diagonal.

Isotropic means "the same in every direction". An isotropic Gaussian has 
- the same variance for every weight
- zero covariance between weights.

### Now I will fast forward the math derivation sections and get to the meat of things (coding implementation)

Here are the important four equations:

1. Posterior precision: $S^{-1}_N = \alpha I + \beta \Phi^T\Phi$
2. Posterior mean: $m_N = \beta S_N \Phi^T t$
3. Predictive mean at a new x: $m^T_N \phi (x)$
4. Predictive variance at a new x: $1/\beta + \phi (x)^T S_N \phi (x)$

In plain terms:

1. Posterior precision $(S^{-1}_N)$ = prior certainty ($\alpha$; $1/\alpha$ is the prior variance) plus certainty from the data ($\beta, \Phi$) = how sure we are about $w$.
2. Posterior mean $m_N$ = the best-guess weights.
3. Predictive mean at a new x = the best-guess curve.
4. Predictive variance at a new x = noise (thermometer wobble) plus our uncertainty about the curve.

## Implementation

### Posterior and posterior predictive distribution

`posterior` computes the mean and covariance matrix of the posterior distribution. Given the readings, update the believed weights.
- In: 
    - the design matrix $\Phi$ (inputs passed through basis functions, one row per reading)
    - the measurements t
    - the two "confidence" knobs: $\alpha$ (prior), $\beta$ (noise)
- Out:
    - `m_N` (best-guess weights)
    - `S_N` (how unsure we still are about them)

`posterior_predictive` computes the mean and the variances of the posterior predictive distribution. Given new inputs (eg, an hour of the day where you have no readings), determine expected temprature and how sure we are.
- In: 
    - $\Phi$ for the *test* inputs (often a fine grid of x for plotting)
    - outputs of function `posterior` function
- Out: 
    - for each test x, a **mean** (curve's height) and a **variance** (how wide the band is there).

These functions together turn **dots -> weights -> curve with a band.**



In [ ]:
import numpy as np


def posterior(Phi, t, alpha, beta, return_inverse=False):
    """Computes mean and covariance matrix of the posterior distribution"""
    S_N_inv = alpha * np.eye(Phi.shape[1]) + beta * Phi.T.dot(Phi)
    S_N = np.linalg.inv(S_N_inv)
    m_N = beta * S_N.dot(Phi.T).dot(t)
    
    if return_inverse:
        return m_N, S_N, S_N_inv
    else:
        return m_N, S_N
    
    
def posterior_predictive(Phi_test, m_N, S_N, beta):
    """Computes mean and variances of the posterior predictive distribution"""
    y = Phi_test.dot(m_N)
    # Only compute variances (diagonal elements of covariance matrix)
    y_var = 1 / beta + np.sum(Phi_test.dot(S_N) * Phi_test, axis=1)
    
    return y, y_var

### Example datasets

In [ ]:
f_w0 = -0.3
f_w1 =  0.5


def f(X, noise_variance):
    '''Linear function plus noise'''
    return f_w0 + f_w1 * X + noise(X.shape, noise_variance)


def g(X, noise_variance):
    '''Sinusoidial function plus noise'''
    return 0.5 + np.sin(2 * np.pi * X) + noise(X.shape, noise_variance)


def noise(size, variance):
    return np.random.normal(scale=np.sqrt(variance), size=size)

### Basis functions

In [ ]:
def identity_basis_function(x):
    return x


def gaussian_basis_function(x, mu, sigma=0.1):
    return np.exp(-0.5 * (x - mu) ** 2 / sigma ** 2)


def polynomial_basis_function(x, power):
    return x ** power


def expand(x, bf, bf_args=None):
    if bf_args is None:
        return np.concatenate([np.ones(x.shape), bf(x)], axis=1)
    else:
        return np.concatenate([np.ones(x.shape)] + [bf(x, bf_arg) for bf_arg in bf_args], axis=1)

### Straight line fitting

For straight-line fitting, we use a linear regression model with only $w_0$, $w_1$. Predictions are made with `posterior_predictive` distribution. Since this model only has two parameters, we can visualize the posterior density in 2D, which is done in the first column after the output. Rows use increasing number of training data from a dataset.

In [ ]:
np.random.seed(0)

from reference.bayesian_linear_regression_util import *

import matplotlib.pyplot as plt
%matplotlib inline

# Training dataset sizes
N_list = [1, 3, 20]

beta = 25.0 # Noise
alpha = 2.0 # Prior

# Training observations in [-1, 1)
X = np.random.rand(N_list[-1], 1) * 2 - 1

# Training target values
t = f(X, noise_variance=1/beta)

# Test observations
X_test = np.linspace(-1, 1, 100).reshape(-1, 1)

# Function values without noise
y_true = f(X_test, noise_variance=0)

# Design matrix of test observations
Phi_test = expand(X_test, identity_basis_function)

plt.figure(figsize=(15, 10))
plt.subplots_adjust(hspace=0.4)

for i, N in enumerate(N_list):
    X_N = X[:N]
    t_N = t[:N]
    
    # Design matrix of training observations
    Phi_N = expand(X_N, identity_basis_function)
    
    # Mean and covariance matrix of posterior
    m_N, S_N = posterior(Phi_N, t_N, alpha, beta)
    
    # Mean and variances of posterior predictive
    y, y_var = posterior_predictive(Phi_test, m_N, S_N, beta)
    
    # Draw 5 random weight samples from posterior and compute y values
    w_samples = np.random.multivariate_normal(m_N.ravel(), S_N, 5).T
    y_samples = Phi_test.dot(w_samples)
    
    plt.subplot(len(N_list), 3, i * 3 + 1)
    plot_posterior(m_N, S_N, f_w0, f_w1)
    plt.title(f'Posterior density (N = {N})')
    plt.legend()
    
    plt.subplot(len(N_list), 3, i * 3 + 2)
    plot_data(X_N, t_N)
    plot_truth(X_test, y_true)
    plot_posterior_samples(X_test, y_samples)
    plt.ylim(-1.5, 1.0)
    plt.legend()
    
    plt.subplot(len(N_list), 3, i * 3 + 3)
    plot_data(X_N, t_N)
    plot_truth(X_test, y_true, label=None)
    plot_predictive(X_test, y, np.sqrt(y_var))
    plt.ylim(-1.5, 1.0)
    plt.legend()

Next: explain this graph ...